In [ ]:
# import modules
import pandas as pd
import re

In [ ]:
# read in the data
df = pd.read_csv('./steam_games_reqs/steam_games_req_data.csv')
df

In [ ]:
# remove ? character from entire table
df = df.replace('?', '', regex=True)

# remove ? character from entire table
df = df.replace('?', '', regex=True)

df

In [ ]:
# first, let's try to clean the operating system column using regular expressions
# let's make sure everything is a string
df['min_req_os'] = df['min_req_os'].astype(str)

# store the data to a list for manipulation outside the dataframe
os_list = df['min_req_os'].tolist()

# to start off, let's see what the unique values are
set(os_list)

In [ ]:
# let's print any unique strings that contain a 'not' specification
for os in list(set(os_list)):
    if 'not' in os:
        print(os)

In [ ]:
[os for os in os_list if '83' in os]

In [ ]:
# create a function for cleaning operating system data
def clean_os_strings(os_list):

    # remove the 'OS *:' and the ? character from any string that has it
    os_list = [re.sub(r'OS \*:|?', '', os).strip() for os in os_list]

    # As we saw in min_req, we need to remove 'not' text: '(32-bit not supported)'
    os_list = [re.sub(r' \(32-bit not supported\)', '', os).strip() for os in os_list]

    # now let's selectively remove text that doesn't correspond to strict operating system relevant strings
    os_list = [' '.join(re.findall(r'Windows|XP|Vista|7\.?[0-9]?|8\.?[0-9]?|10\.?[0-9]?|11\.?[0-9]?|32|64|nan', os, re.IGNORECASE)).strip() for os in os_list]

    # define a list of regular strings as the standard order for operating systems, in order of release (oldest to newest)
    release_order = [r'XP', r'Vista', r'[^0-9]7\.?[0-9]?[^0-9]', r'[^0-9]8\.?[0-9]?[^0-9]', r'[^0-9]10\.?[0-9]?[^0-9]', r'[^0-9]11\.?[0-9]?[^0-9]']

    # same for bit architecture
    bit_order = [r'32', r'64']

    # create empty list to store cleaned strings
    cleaned_os_list = []

    # loop over all strings
    for os in os_list:

        # First check for Windows
        match_brand = re.search(r'Windows', os, re.IGNORECASE)
        if match_brand: os_brand = match_brand.group(0).strip().lower()

        # if no brand is found, assume is Windows
        else: os_brand = 'windows'
        

        # Now check for the specific os release (loop through them, break when a match is found)
        os_release = None
        for release in release_order:
            match_release = re.search(release, os, re.IGNORECASE)
            if match_release:
                os_release = match_release.group(0).strip()
                break

        # If no release is found, assume minimum is Windows 10
        if not os_release: os_release = '10'

        # Lastly, check for the bit architecture
        os_bit = None
        for bit in bit_order:
            match_bit = re.search(bit, os, re.IGNORECASE)
            if match_bit:
                os_bit = match_bit.group(0).strip()
                break

        # If no bit is found, assume minimum is 32 bit
        if not os_bit: os_bit = '32'

        # combine the elements into a single string and append
        cleaned_os_list.append(f'{os_brand} {os_release} {os_bit}')

    return cleaned_os_list

In [ ]:
# use the function to clean the operating system columns of the dataframe
df['min_req_os'] = clean_os_strings(os_list)
df['rec_req_os'] = clean_os_strings(df['rec_req_os'].astype(str).tolist())
df

In [ ]:
# create a function to clean the graphics card data
def clean_graph_data(graph_list):

    # create empty list to store cleaned strings
    clean_strings_nvidia = []
    clean_strings_amd    = []

    # loop over all strings
    for graph in graph_list:

        # remove the string 'GPU' from any entry that has it
        string = re.sub(r' ?GPU ?', ' ', str(graph), re.IGNORECASE)
        # print(string)

        # define the regular expression patterns for NVIDIA graphics cards
        nvidia_pattern_brand  = r'(^| |/)(?P<brand>nvidia)  *(?P<series>ge?force)? *(?P<prefix>gtx|rtx|gts|gt)? *(?P<model>[0-9]+)? *(?P<suffix>ti|gt|gs)?'
        nvidia_pattern_series =                     r'(^| |/)(?P<series>ge?force) *(?P<prefix>gtx|rtx|gts|gt)? *(?P<model>[0-9]+)? *(?P<suffix>ti|gt|gs)?'
        nvidia_pattern_prefix =                                           r'(^| |/)(?P<prefix>gtx|rtx|gts|gt) *(?P<model>[0-9]+)? *(?P<suffix>ti|gt|gs)?'

        # search for the nvidia patterns in the string
        nvidia_match_brand  = re.search(nvidia_pattern_brand , string, re.IGNORECASE)
        nvidia_match_series = re.search(nvidia_pattern_series, string, re.IGNORECASE)
        nvidia_match_prefix = re.search(nvidia_pattern_prefix, string, re.IGNORECASE)
        
        # use the first match that is not None from above
        nvidia_match = nvidia_match_brand if nvidia_match_brand else nvidia_match_series if nvidia_match_series else nvidia_match_prefix if nvidia_match_prefix else None

        # store the match groups in a dictionary
        nvidia_dict = nvidia_match.groupdict() if nvidia_match else {'brand': None, 'series': None, 'prefix': None, 'model': None, 'suffix': None}

        # if 2 or more dictionary values are not None (avoiding false positives with relaxed search grabbing only a number)
        if sum([val is not None for val in nvidia_dict.values()]) >= 2:

            # update missing prefix data
            if nvidia_dict['prefix'] is None:

                # if model exixsts
                if nvidia_dict['model'] is not None:

                    # if the model is 5000 or higher
                    if int(nvidia_dict['model']) >= 7000:

                        # if there is no suffix, assume GT
                        if nvidia_dict['suffix'] is None:
                            nvidia_dict['suffix'] = 'GT'
                    
                    # else if model is less than 2000, assume GTX
                    elif int(nvidia_dict['model']) < 2000:
                        nvidia_dict['prefix'] = 'GTX'

                    # else if model is less than 5000, assume RTX
                    elif int(nvidia_dict['model']) < 5000:
                        nvidia_dict['prefix'] = 'RTX'

            # all modern cards are GeForce, and everything here is NVIDIA
            nvidia_dict['brand' ] = 'NVIDIA'
            nvidia_dict['series'] = 'GeForce'

            # add nvidia card to the clean string for this entry
            clean_string_nvidia = ' '.join([str(nvidia_dict['brand']), str(nvidia_dict['series']), str(nvidia_dict['prefix']), str(nvidia_dict['model']), str(nvidia_dict['suffix'])]).replace(' None', '')

        # else there was no NVIDIA match, initialize clean string as empty
        else: clean_string_nvidia = ''

        # define the regular expression patterns for AMD graphics cards
        amd_pattern_brand  = r'(^| |/)(?P<brand>amd)  *(?P<series>radeon)? *(?P<prefix>r[a-z0-9]|hd|x)? *(?P<model>[0-9]+)? *(?P<suffix>xt|pro|xx|x|le)?'
        amd_pattern_series =                  r'(^| |/)(?P<series>radeon) *(?P<prefix>r[a-z0-9]|hd|x)? *(?P<model>[0-9]+)? *(?P<suffix>xt|pro|xx||le)?'
        amd_pattern_prefix =                                      r'(^| |/)(?P<prefix>r[x0-9]) *(?P<model>[0-9]+)? *(?P<suffix>xt|pro|xx||le)?'

        # search for the amd patterns in the string
        amd_match_brand  = re.search(amd_pattern_brand , string, re.IGNORECASE)
        amd_match_series = re.search(amd_pattern_series, string, re.IGNORECASE)
        amd_match_prefix = re.search(amd_pattern_prefix, string, re.IGNORECASE)

        # use the first match that is not None from above
        amd_match = amd_match_brand if amd_match_brand else amd_match_series if amd_match_series else amd_match_prefix if amd_match_prefix else None

        # store the match groups in a dictionary
        amd_dict = amd_match.groupdict() if amd_match else {'brand': None, 'series': None, 'prefix': None, 'model': None, 'suffix': None}

        # if 2 or more dictionary values are not None (avoiding false positives with relaxed search grabbing only a number) and the model is not None
        if sum([val is not None for val in amd_dict.values()]) >= 2 and amd_dict['model'] is not None:

            # if the suffix is 'XX', replace prefix with 'HD' and assume model is [0-9]{2}00
            if amd_dict['suffix'] == 'XX':
                amd_dict['prefix'] = 'HD'
                amd_dict['model']  = f'{amd_dict["model"]}00'

            # all modern cards are Radeon, and everything here is AMD
            amd_dict['brand' ] = 'AMD'
            amd_dict['series'] = 'Radeon'

            # add amd card to the clean string for this entry
            clean_string_amd = ' '.join([str(amd_dict['brand']), str(amd_dict['series']), str(amd_dict['prefix']), str(amd_dict['model']), str(amd_dict['suffix'])]).replace(' None', '')

        # else there was no AMD match, initialize clean string as empty
        else: clean_string_amd = ''

        # append the clean string to the list of clean strings
        clean_strings_nvidia.append(clean_string_nvidia.lower())
        clean_strings_amd.append(   clean_string_amd.lower())
        # print(clean_string)
        # print()

    return (clean_strings_nvidia, clean_strings_amd)


In [ ]:
# use the graphics card cleaning function to clean the minimum and recommended graphics card columns
df['min_req_graph_nvidia'], df['min_req_graph_amd'] = clean_graph_data(df['min_req_graph'].astype(str).tolist())
df['rec_req_graph_nvidia'], df['rec_req_graph_amd'] = clean_graph_data(df['rec_req_graph'].astype(str).tolist())

# remove the original graphics card columns
df = df.drop(columns=['min_req_graph', 'rec_req_graph'])

# reorder the columns alphabetically
df = df.reindex(sorted(df.columns), axis=1)
df

In [ ]:
# get all unique values for min_req_dirX
set(df['min_req_dirX'])

In [ ]:
# function to clean the directX data
def clean_dirX_data(dirX_list):

    # create empty list to store cleaned strings
    clean_strings = []

    # loop over all strings
    for string in dirX_list:

        # define the regular expression patterns for directX
        dirX_pattern_dirX = r'(^|[^a-z0-9])(directx)-? *(version)? *(?P<version>[0-9]+\.?[0-9]?[a-z]?)'
        dirX_pattern_vers =              r'(^|[^a-z0-9])(version) *(?P<version>[0-9]+\.?[0-9]?[a-z]?)'

        # search for the directX patterns in the string
        dirX_match_dirX = re.search(dirX_pattern_dirX, string, re.IGNORECASE)
        dirX_match_vers = re.search(dirX_pattern_vers, string, re.IGNORECASE)

        # use the first match that is not None from above
        dirX_match = dirX_match_dirX if dirX_match_dirX else dirX_match_vers if dirX_match_vers else None

        # store the match groups in a dictionary
        dirX_dict = dirX_match.groupdict() if dirX_match else {'version': None}

        # if version was found, add to clean string
        if dirX_dict['version'] is not None: clean_string = ' '.join(['version', str(dirX_dict["version"])])

        # else there was no directX match, initialize clean string as empty
        else: clean_string = ''

        # append the clean string to the list of clean strings
        clean_strings.append(clean_string)
        # print(string)
        # print(clean_string)
        # print()

    return clean_strings

In [ ]:
# use the directX cleaning function to clean the minimum and recommended directX columns
df['min_req_dirX'] = clean_dirX_data(df['min_req_dirX'].astype(str).tolist())
df['rec_req_dirX'] = clean_dirX_data(df['rec_req_dirX'].astype(str).tolist())
df

In [ ]:
# function to clean the storage data
def clean_storage_data(storage_list):

    # create empty list to store cleaned strings
    clean_strings = []

    # loop over all strings
    for string in storage_list:

        # define the regular expression patterns for storage
        storage_pattern = r'(?P<size>[0-9]+) *(?P<unit>mb|gb|tb)'

        # search for the storage patterns in the string
        storage_match = re.search(storage_pattern, string, re.IGNORECASE)

        # store the match groups in a dictionary
        storage_dict = storage_match.groupdict() if storage_match else {'size': None, 'unit': None}

        # if size and unit were found, add to clean string
        if storage_dict['size'] is not None and storage_dict['unit'] is not None: clean_string = ' '.join([str(storage_dict["size"]), str(storage_dict["unit"])])
        
        # else there was no storage match, initialize clean string as empty
        else: clean_string = ''

        # append the clean string to the list of clean strings
        clean_strings.append(clean_string)
        # print(string)
        # print(clean_string)
        # print()

    return clean_strings

In [ ]:
# use the storage cleaning function to clean the minimum and recommended storage columns
df['min_req_stor'] = clean_storage_data(df['min_req_stor'].astype(str).tolist())
df['rec_req_stor'] = clean_storage_data(df['rec_req_stor'].astype(str).tolist())
df

In [ ]:
# create a function to clean the memory data
def clean_mem_data(mem_list):

    # create empty list to store cleaned strings
    clean_strings = []

    # loop over all strings
    for string in mem_list:

        # define the regular expression patterns for memory
        mem_pattern = r'(?P<size>[0-9]+) *(?P<unit>mb|gb)'

        # search for the memory patterns in the string
        mem_match = re.search(mem_pattern, string, re.IGNORECASE)

        # store the match groups in a dictionary
        mem_dict = mem_match.groupdict() if mem_match else {'size': None, 'unit': None}

        # if size and unit were found, add to clean string
        if mem_dict['size'] is not None and mem_dict['unit'] is not None: clean_string = ' '.join([str(mem_dict["size"]), str(mem_dict["unit"])])
        
        # else there was no memory match, initialize clean string as empty
        else: clean_string = ''

        # append the clean string to the list of clean strings
        clean_strings.append(clean_string)
        # print(string)
        # print(clean_string)
        # print()

    return clean_strings

In [ ]:
# use the memory cleaning function to clean the minimum and recommended memory columns
df['min_req_mem'] = clean_mem_data(df['min_req_mem'].astype(str).tolist())
df['rec_req_mem'] = clean_mem_data(df['rec_req_mem'].astype(str).tolist())
df

In [ ]:
# create a function to clean the cpu data
def clean_cpu_data(cpu_list):

    # create empty list to store cleaned strings
    clean_strings_intel = []
    clean_strings_amd   = []

    # loop over all strings
    for string in cpu_list:

        # define the regular expression patterns for Intel CPUs
        intel_pattern_brand  = r'(^| |/)(?P<brand>intel)  *(?P<series>core)? *(?P<prefix>i[0-9])-? *(?P<model>[0-9][0-9]+)(?P<suffix>[a-z])?'
        intel_pattern_series =                    r'(^| |/)(?P<series>core) *(?P<prefix>i[0-9])-? *(?P<model>[0-9][0-9]+)(?P<suffix>[a-z])?'
        intel_pattern_prefix =                                      r'(^| |/)(?P<prefix>i[0-9])-? *(?P<model>[0-9][0-9]+)(?P<suffix>[a-z])?'

        # search for the intel patterns in the string
        intel_match_brand  = re.search(intel_pattern_brand , string, re.IGNORECASE)
        intel_match_series = re.search(intel_pattern_series, string, re.IGNORECASE)
        intel_match_prefix = re.search(intel_pattern_prefix, string, re.IGNORECASE)
        
        # use the first match that is not None from above
        intel_match = intel_match_brand if intel_match_brand else intel_match_series if intel_match_series else intel_match_prefix if intel_match_prefix else None

        # store the match groups in a dictionary
        intel_dict = intel_match.groupdict() if intel_match else {'brand': None, 'series': None, 'prefix': None, 'model': None, 'suffix': None}

        # if 2 or more dictionary values are not None (avoiding false positives with relaxed search grabbing only a number)
        if sum([val is not None for val in intel_dict.values()]) >= 2:

            # all modern intel CPUs are Core, and everything here is intel
            intel_dict['brand' ] = 'intel'
            intel_dict['series'] = 'core'

            # add intel card to the clean string for this entry
            clean_string_intel = ' '.join([str(intel_dict['brand']), str(intel_dict['series']), str(intel_dict['prefix']), str(intel_dict['model']), str(intel_dict['suffix'])]).replace(' None', '')

        # else there was no intel match, initialize clean string as empty
        else: clean_string_intel = ''

        # define the regular expression patterns for AMD graphics cards
        amd_pattern_brand  = r'(^| |/)(?P<brand>amd)  *(?P<series>ryzen|fx|phenom ?i?i?)?-? *(?P<prefix>[^0-9][0-9][^0-9]|x[0-9])? *(?P<model>[0-9][0-9]+)(?P<suffix>[a-z])?'
        amd_pattern_series =                  r'(^| |/)(?P<series>ryzen|fx|phenom ?i?i?)-? *(?P<prefix>[^0-9][0-9][^0-9]|x[0-9])? *(?P<model>[0-9][0-9]+)(?P<suffix>[a-z])?'

        # search for the amd patterns in the string
        amd_match_brand  = re.search(amd_pattern_brand , string, re.IGNORECASE)
        amd_match_series = re.search(amd_pattern_series, string, re.IGNORECASE)

        # use the first match that is not None from above
        amd_match = amd_match_brand if amd_match_brand else amd_match_series if amd_match_series else None

        # store the match groups in a dictionary
        amd_dict = amd_match.groupdict() if amd_match else {'brand': None, 'series': None, 'prefix': None, 'model': None, 'suffix': None}

        # if 2 or more dictionary values are not None (avoiding false positives with relaxed search grabbing only a number)
        if sum([val is not None for val in amd_dict.values()]) >= 2:

            # everything here is AMD
            amd_dict['brand' ] = 'AMD'

            # add amd card to the clean string for this entry
            clean_string_amd = ' '.join([str(amd_dict['brand']).strip(), str(amd_dict['series']).strip(), str(amd_dict['prefix']).strip(), str(amd_dict['model']).strip(), str(amd_dict['suffix']).strip()]).replace(' None', '')

        # else there was no AMD match, initialize clean string as empty
        else: clean_string_amd = ''

        # append the clean string to the list of clean strings
        clean_strings_intel.append(clean_string_intel.lower())
        clean_strings_amd.append(   clean_string_amd.lower())
        # print(string)
        # print(clean_string)
        # print()

    return (clean_strings_intel, clean_strings_amd)

In [ ]:
# use the CPU cleaning function to clean the minimum and recommended CPU columns
df['min_req_proc_intel'], df['min_req_proc_amd'] = clean_cpu_data(df['min_req_proc'].astype(str).tolist())
df['rec_req_proc_intel'], df['rec_req_proc_amd'] = clean_cpu_data(df['rec_req_proc'].astype(str).tolist())

# remove the original CPU columns
df = df.drop(columns=['min_req_proc', 'rec_req_proc'])

# reorder the columns alphabetically
df = df.reindex(sorted(df.columns), axis=1)
df

In [ ]:
# reorder columns like [rank, name, min_*, rec_*]
df = df[['rank', 'name'] + [col for col in df.columns if 'min' in col] + [col for col in df.columns if 'rec' in col]]
df

In [ ]:
# write the cleaned dataframe to a new csv file
df.to_csv('./steam_games_reqs/steam_games_req_data_cleaned.csv', index=False)